# WCF sensitivity r50 rerun, shard 38 of 120

This shard contains 54 estimator cells grouped into 25 paired replications of the frozen sensitivity roster. Its conservative reference estimate is 6.99 hours; Colab is slower than the reference machine, so run it in a fresh session and let the checkpoint cell resume it if the session ends. Run all cells.

In [ ]:
import os
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
             'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS', 'R_NUM_THREADS'):
    os.environ[name] = '1'
print('numerical libraries pinned to one thread')


In [ ]:
import subprocess, sys
PYTHON_PACKAGES = ('numpy==2.4.3', 'scipy==1.17.1', 'scikit-learn==1.8.0', 'pandas==3.0.1', 'pyarrow==24.0.0')
completed = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', *PYTHON_PACKAGES],
    text=True, capture_output=True,
)
if completed.returncode != 0:
    print(completed.stdout[-2000:])
    raise SystemExit(completed.stderr[-4000:])
print('Python dependencies installed:', ', '.join(PYTHON_PACKAGES))
print('the kernel keeps its pre-imported stack; numerical work runs in a '
      'fresh child process that loads the pins from disk')


In [ ]:
import base64, hashlib, os, pathlib, sys, tempfile, zipfile
SOURCE_ARCHIVE_SHA256 = '4ecccf6b34997f79761d20769db1351eb5aad854dcf6ce5d46beb55f8a3191d9'
SOURCE_ARCHIVE_B64 = '''\
...
'''
workdir = pathlib.Path(tempfile.mkdtemp(prefix='wcf_sensitivity_r50_'))
archive_path = workdir / 'source.zip'
archive_path.write_bytes(base64.b64decode(SOURCE_ARCHIVE_B64))
assert hashlib.sha256(archive_path.read_bytes()).hexdigest() == SOURCE_ARCHIVE_SHA256
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(workdir)
sys.path[:0] = [str(workdir), str(workdir / 'src')]
os.environ['WCF_SOURCE_ROOT'] = str(workdir)
print('source archive:', SOURCE_ARCHIVE_SHA256)
print('manifest:', '94c31a64332f2940d3b4fcd0619349b2b2234c88a9bd924dbe314a9d3611c471')


In [ ]:
# The R bridge subprocesses inherit this variable from the notebook process, so
# it must be set in Python rather than in the install shell below.
import os
os.environ['WCF_CAUSAL_DRF_R_LIB'] = '/content/Rlib/causal_drf'
print('WCF_CAUSAL_DRF_R_LIB =', os.environ['WCF_CAUSAL_DRF_R_LIB'])


In [ ]:
%%bash
set -e
apt-get -qq update > /dev/null 2>&1
apt-get -qq install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev curl > /dev/null 2>&1
Rscript -e 'options(Ncpus=2); install.packages(c("Rcpp","RcppEigen","jsonlite","remotes","transport","fastDummies","kernlab"), repos="https://cloud.r-project.org", quiet=TRUE)'
Rscript -e 'options(Ncpus=2); remotes::install_version("drf", version="1.3.1", repos="https://cloud.r-project.org", upgrade="never", quiet=TRUE); stopifnot(as.character(packageVersion("drf")) == "1.3.1")'
mkdir -p /content/Rlib/causal_drf
CAUSAL_SHA="0a1a508444176b5b1553f13e832be93a374b0af2"
TARBALL="/tmp/causal_clean_${CAUSAL_SHA:0:12}.tar.gz"
EXTRACT="/tmp/causal_clean_${CAUSAL_SHA:0:12}"
curl -fsSL "https://codeload.github.com/herbps10/drf/tar.gz/${CAUSAL_SHA}" -o "$TARBALL"
rm -rf "$EXTRACT"
mkdir -p "$EXTRACT"
tar -xzf "$TARBALL" -C "$EXTRACT"
PKG_DIR=$(find "$EXTRACT" -maxdepth 4 -type d -path "*/r-package/drf" | head -1)
test -n "$PKG_DIR"
R CMD INSTALL --library=/content/Rlib/causal_drf "$PKG_DIR"
Rscript -e '.libPaths(c("/content/Rlib/causal_drf",.libPaths())); stopifnot(requireNamespace("drf", quietly=TRUE)); cat("causal-clean", as.character(packageVersion("drf")), "ready\n")'
echo 'R dependencies installed and validated'


In [ ]:
# Pin the authors' Causal-DRF simulation repository. The model is implemented
# by the causal-clean `drf` fork installed above; this cell pins the reference
# repository whose `drf()` call and hyperparameters the driver reproduces.
import hashlib, re, urllib.request
CAUSAL_DRF_PAPER_REPOSITORY = 'herbps10/causal_drf_paper'
CAUSAL_DRF_PAPER_COMMIT = '06d156e1f2c17c676000f258ccdf15fc60544384'
CAUSAL_CLEAN_DRF_COMMIT = '0a1a508444176b5b1553f13e832be93a374b0af2'
CAUSAL_DRF_PAPER_FILES = {
    'R/simulation_study_setup.R': 'b4b6f94054c3814c7cce272cd66a10bf772a2efbcdeb98ef058e425a5f2607a2',
    'R/simulation_study.R': 'fbf7ec12a375b169737edaa07e4c3bb62bf5d10bcf56f80a66bbc307a2778a59',
}
CAUSAL_DRF_PAPER_BASE = (
    'https://raw.githubusercontent.com/' + CAUSAL_DRF_PAPER_REPOSITORY + '/'
    + CAUSAL_DRF_PAPER_COMMIT + '/'
)
CAUSAL_DRF_PAPER_SOURCES = {}
for path, expected in CAUSAL_DRF_PAPER_FILES.items():
    payload = urllib.request.urlopen(
        CAUSAL_DRF_PAPER_BASE + path, timeout=120
    ).read()
    digest = hashlib.sha256(payload).hexdigest()
    assert digest == expected, (path, digest, expected)
    CAUSAL_DRF_PAPER_SOURCES[path] = payload.decode('utf-8')
    print('verified', CAUSAL_DRF_PAPER_REPOSITORY + '@' + CAUSAL_DRF_PAPER_COMMIT,
          path, digest)

setup_source = re.sub(r'\s+', ' ', CAUSAL_DRF_PAPER_SOURCES['R/simulation_study_setup.R'])
study_source = re.sub(r'\s+', ' ', CAUSAL_DRF_PAPER_SOURCES['R/simulation_study.R'])
for fragment in (
    'drf(X, Y, W, num.trees = num_trees, ci.group.size = ci_group_size',
    'response.scaling = FALSE',
):
    assert fragment in setup_source, fragment
for fragment in (
    'dgp = c("nothing", "confounding", "effect", "both")',
    'num_trees = c(50 * 50)',
    'ci_group_size = round(num_trees / 50)',
):
    assert fragment in study_source, fragment
print('Causal-DRF baseline pin verified: causal-clean drf fork at',
      CAUSAL_CLEAN_DRF_COMMIT)
print('the WCF driver reproduces the pinned call with num_trees=2500, '
      'ci.group.size=50, response.scaling=FALSE (seed explicit, one thread)')


In [ ]:
import json
from pathlib import Path
SHARD_INDEX = 38
SHARD_TOTAL = 120
ESTIMATED_REFERENCE_SECONDS = 25155.0
MANIFEST_SLICE = json.loads('''{"manifest_contract_id": "WCF-SENSITIVITY-R50-v1", "manifest_checksum": "94c31a64332f2940d3b4fcd0619349b2b2234c88a9bd924dbe314a9d3611c471", "estimator_source_hash": "9379c84445660650ced9e93d926eec3996c5662ea601544400b610cc0cd4b515", "evaluation_protocol_id": "WCF-CONFIRMATORY-EVAL-v1", "method_registry": {"cwdb_dr": {"role": "variant", "adapter": "cwdb_dr", "produces_law": true, "cross_fitted": true, "parameters": {"contrast_candidates": [0.0, 50.0, 500.0], "n_folds": 3, "common_grid_levels": "", "propensity_factory": "logistic"}, "propensity_clip": [0.02, 0.98], "boosting_budget": {"n_estimators": 100, "learning_rate": 0.12, "max_depth": 4, "min_samples_leaf": 10, "min_arm_leaf": 5, "collision_epsilon": 0.001}, "estimator_source_hash": "9379c84445660650ced9e93d926eec3996c5662ea601544400b610cc0cd4b515"}, "cwdb_dr_flex": {"role": "variant", "adapter": "cwdb_dr", "produces_law": true, "cross_fitted": true, "parameters": {"contrast_candidates": [0.0, 50.0, 500.0], "n_folds": 3, "common_grid_levels": "", "propensity_factory": "hist_gradient_boosting"}, "propensity_clip": [0.02, 0.98], "boosting_budget": {"n_estimators": 100, "learning_rate": 0.12, "max_depth": 4, "min_samples_leaf": 10, "min_arm_leaf": 5, "collision_epsilon": 0.001}, "estimator_source_hash": "9379c84445660650ced9e93d926eec3996c5662ea601544400b610cc0cd4b515"}, "cwdb_dr_oracle": {"role": "diagnostic", "adapter": "cwdb_dr", "produces_law": true, "cross_fitted": true, "parameters": {"contrast_candidates": [0.0, 50.0, 500.0], "n_folds": 3, "common_grid_levels": "", "oracle_propensity": true}, "propensity_clip": [0.02, 0.98], "boosting_budget": {"n_estimators": 100, "learning_rate": 0.12, "max_depth": 4, "min_samples_leaf": 10, "min_arm_leaf": 5, "collision_epsilon": 0.001}, "estimator_source_hash": "9379c84445660650ced9e93d926eec3996c5662ea601544400b610cc0cd4b515"}, "causal_drf": {"role": "baseline", "adapter": "forest", "produces_law": true, "parameters": {"method": "causal_drf"}}, "drf": {"role": "baseline", "adapter": "forest", "produces_law": true, "parameters": {"method": "drf"}}, "cwdb_dr_flex_rf": {"role": "variant", "adapter": "cwdb_dr", "produces_law": true, "cross_fitted": true, "parameters": {"contrast_candidates": [0.0, 50.0, 500.0], "n_folds": 3, "common_grid_levels": "", "propensity_factory": "random_forest"}, "estimator_source_hash": "9379c84445660650ced9e93d926eec3996c5662ea601544400b610cc0cd4b515"}}, "cells": [{"grid": "wcf_sensitivity_v1_align", "dgp": "SYM-ALIGN", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 18, "cell_key": "506ba41c1bb58f8a", "test_seed": 900018}, {"grid": "wcf_sensitivity_v1_align", "dgp": "SYM-ALIGN", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 18, "cell_key": "48edea05c7944f8a", "test_seed": 900018}, {"grid": "wcf_sensitivity_v1_align", "dgp": "SYM-ALIGN", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 18, "cell_key": "88f3534f2a304478", "test_seed": 900018}, {"grid": "wcf_sensitivity_v1_align", "dgp": "SYM-IRREL", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 48, "cell_key": "b7f40fd734e6345b", "test_seed": 900048}, {"grid": "wcf_sensitivity_v1_align", "dgp": "SYM-IRREL", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 48, "cell_key": "d9b03fd66e7e13e2", "test_seed": 900048}, {"grid": "wcf_sensitivity_v1_align", "dgp": "SYM-IRREL", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 48, "cell_key": "2712a4576e024f07", "test_seed": 900048}, {"grid": "wcf_sensitivity_v1_align_null", "dgp": "SYM-ALIGN-NULL", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 38, "cell_key": "1aad41e1aef8ade7", "test_seed": 900038}, {"grid": "wcf_sensitivity_v1_align_null", "dgp": "SYM-ALIGN-NULL", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 38, "cell_key": "cacbdb62ff1a4584", "test_seed": 900038}, {"grid": "wcf_sensitivity_v1_align_null", "dgp": "SYM-ALIGN-NULL", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 38, "cell_key": "91a017f7c8890d98", "test_seed": 900038}, {"grid": "wcf_sensitivity_v1_flex", "dgp": "SYM-MU", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr_flex", "seed": 38, "cell_key": "70ff617912296800", "test_seed": 900038}, {"grid": "wcf_sensitivity_v1_flex", "dgp": "SYM-MU", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr_oracle", "seed": 38, "cell_key": "7f29f3478d407040", "test_seed": 900038}, {"grid": "wcf_sensitivity_v1_flex_rf", "dgp": "SYM-NL", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr_flex_rf", "seed": 8, "cell_key": "3184fa7a32b5f14c", "test_seed": 900008}, {"grid": "wcf_sensitivity_v1_flex_rf", "dgp": "SYM-NL", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr_flex_rf", "seed": 28, "cell_key": "ca1373e10730dc92", "test_seed": 900028}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 5, "method": "cwdb_dr", "seed": 18, "cell_key": "9337ed9b3bf32d84", "test_seed": 900018}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 18, "cell_key": "80ec8581d7c45b85", "test_seed": 900018}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 18, "cell_key": "fc09bf2b0c99e7e9", "test_seed": 900018}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 18, "cell_key": "f2c3183ff96cc37a", "test_seed": 900018}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC1", "n_train": 1000, "n_grid": 5, "n_particles": 10, "method": "cwdb_dr", "seed": 28, "cell_key": "5d9c94f7327c3ed6", "test_seed": 900028}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC1", "n_train": 1000, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 28, "cell_key": "280b6b74ddb18434", "test_seed": 900028}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC1", "n_train": 1000, "n_grid": 5, "n_particles": 10, "method": "drf", "seed": 28, "cell_key": "1882f573f2055412", "test_seed": 900028}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC1", "n_train": 1000, "n_grid": 5, "n_particles": 25, "method": "cwdb_dr", "seed": 18, "cell_key": "edffec2858368c72", "test_seed": 900018}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 25, "method": "cwdb_dr", "seed": 28, "cell_key": "d0baf5943d399cbe", "test_seed": 900028}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC1", "n_train": 1000, "n_grid": 49, "n_particles": 5, "method": "cwdb_dr", "seed": 18, "cell_key": "00f05e8f92721f98", "test_seed": 900018}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC1", "n_train": 1000, "n_grid": 49, "n_particles": 10, "method": "cwdb_dr", "seed": 48, "cell_key": "6b5670cebdd03197", "test_seed": 900048}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC1", "n_train": 1000, "n_grid": 49, "n_particles": 10, "method": "causal_drf", "seed": 48, "cell_key": "40f35e8fb4e71db7", "test_seed": 900048}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC1", "n_train": 1000, "n_grid": 49, "n_particles": 10, "method": "drf", "seed": 48, "cell_key": "51632bfd2196430e", "test_seed": 900048}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC1", "n_train": 1000, "n_grid": 49, "n_particles": 25, "method": "cwdb_dr", "seed": 38, "cell_key": "bfb5418179e4bf6e", "test_seed": 900038}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 5, "method": "cwdb_dr", "seed": 38, "cell_key": "2526c41a914b4547", "test_seed": 900038}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 38, "cell_key": "101171c25a14773b", "test_seed": 900038}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 38, "cell_key": "a28312de1cb5869b", "test_seed": 900038}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 38, "cell_key": "009e499242938b7e", "test_seed": 900038}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC3", "n_train": 1000, "n_grid": 5, "n_particles": 5, "method": "cwdb_dr", "seed": 8, "cell_key": "265a8db77b22c163", "test_seed": 900008}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC3", "n_train": 1000, "n_grid": 5, "n_particles": 10, "method": "cwdb_dr", "seed": 48, "cell_key": "df629f452d19fd7d", "test_seed": 900048}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC3", "n_train": 1000, "n_grid": 5, "n_particles": 10, "method": "causal_drf", "seed": 48, "cell_key": "6438db86f537e171", "test_seed": 900048}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC3", "n_train": 1000, "n_grid": 5, "n_particles": 10, "method": "drf", "seed": 48, "cell_key": "a39efa914b88d662", "test_seed": 900048}, {"grid": "wcf_sensitivity_v1_logit_f3", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 25, "method": "cwdb_dr", "seed": 48, "cell_key": "1f76a0684f8e2bfc", "test_seed": 900048}, {"grid": "wcf_sensitivity_v1_sym", "dgp": "SYM-MU", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 8, "cell_key": "16d10657c0cd5c76", "test_seed": 900008}, {"grid": "wcf_sensitivity_v1_sym", "dgp": "SYM-MU", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "7bcf42a42c046da2", "test_seed": 900008}, {"grid": "wcf_sensitivity_v1_sym", "dgp": "SYM-MU", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 8, "cell_key": "086c9fb6064542bd", "test_seed": 900008}, {"grid": "wcf_sensitivity_v1_sym", "dgp": "SYM-MU", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 8, "cell_key": "2c1a414755cd8771", "test_seed": 900008}, {"grid": "wcf_sensitivity_v1_sym", "dgp": "SYM-MU", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "69746f2ece7538d1", "test_seed": 900008}, {"grid": "wcf_sensitivity_v1_sym", "dgp": "SYM-MU", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 8, "cell_key": "0b5049ad56955f28", "test_seed": 900008}, {"grid": "wcf_sensitivity_v1_sym", "dgp": "SYM-RANDOM", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 28, "cell_key": "dfe8e81ba17a9dd3", "test_seed": 900028}, {"grid": "wcf_sensitivity_v1_sym", "dgp": "SYM-RANDOM", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 28, "cell_key": "5d47ed68f146cbf8", "test_seed": 900028}, {"grid": "wcf_sensitivity_v1_sym", "dgp": "SYM-RANDOM", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 28, "cell_key": "d5131b4b40c42225", "test_seed": 900028}, {"grid": "wcf_sensitivity_v1_sym", "dgp": "SYM-RANDOM", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 28, "cell_key": "cc104397d0431894", "test_seed": 900028}, {"grid": "wcf_sensitivity_v1_sym", "dgp": "SYM-RANDOM", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 28, "cell_key": "8a11ef621ca4a207", "test_seed": 900028}, {"grid": "wcf_sensitivity_v1_sym", "dgp": "SYM-RANDOM", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 28, "cell_key": "d5de615f0ffe4328", "test_seed": 900028}, {"grid": "wcf_sensitivity_v1_sym_null", "dgp": "SYM-MU-NULL", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 48, "cell_key": "4a7cb176a4b04abe", "test_seed": 900048}, {"grid": "wcf_sensitivity_v1_sym_null", "dgp": "SYM-MU-NULL", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 48, "cell_key": "76cc3ad9a1719eb1", "test_seed": 900048}, {"grid": "wcf_sensitivity_v1_sym_null", "dgp": "SYM-MU-NULL", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 48, "cell_key": "761477ff2106179f", "test_seed": 900048}, {"grid": "wcf_sensitivity_v1_sym_null", "dgp": "SYM-MU-NULL", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 48, "cell_key": "7edc434f2dcc78e3", "test_seed": 900048}, {"grid": "wcf_sensitivity_v1_sym_null", "dgp": "SYM-MU-NULL", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 48, "cell_key": "eceba074bfb936d8", "test_seed": 900048}, {"grid": "wcf_sensitivity_v1_sym_null", "dgp": "SYM-MU-NULL", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 48, "cell_key": "9d732ed5d833050b", "test_seed": 900048}]}''')
payload = {
    'shard_index': SHARD_INDEX,
    'shard_total': SHARD_TOTAL,
    'manifest_slice': MANIFEST_SLICE,
    'source_archive_sha256': SOURCE_ARCHIVE_SHA256,
    'causal_drf_paper_repository': CAUSAL_DRF_PAPER_REPOSITORY,
    'causal_drf_paper_commit': CAUSAL_DRF_PAPER_COMMIT,
    'causal_drf_paper_files': dict(CAUSAL_DRF_PAPER_FILES),
    'causal_clean_drf_commit': CAUSAL_CLEAN_DRF_COMMIT,
}
output = Path('shard_output')
output.mkdir(exist_ok=True)
Path('wcf_sensitivity_r50_payload.json').write_text(json.dumps(payload), encoding='utf-8')
print('shard', SHARD_INDEX, 'of', SHARD_TOTAL, '| cells', len(MANIFEST_SLICE['cells']))


In [ ]:
import os, subprocess, sys
from pathlib import Path
WORKER_SOURCE = '\\\nimport gc, json, os, sys, time\nfrom pathlib import Path\n\nROOT = os.path.dirname(os.path.abspath(__file__))\nsys.path[:0] = [ROOT, os.path.join(ROOT, \'src\')]\n\nimport pyarrow.parquet as pq\nfrom research.run_wcf_sensitivity_r50 import run_cell\nfrom wasserstein_causal_forests.g3.manifest import Cell\nfrom wasserstein_causal_forests.g3.sensitivity_dgps import register_sensitivity_dgps\nfrom wasserstein_causal_forests.g3.sensitivity_methods import register_sensitivity_methods\nfrom wasserstein_causal_forests.g3.wcf_sensitivity import apply_method_registry\n\nPAYLOAD = json.loads(Path(\'wcf_sensitivity_r50_payload.json\').read_text(encoding=\'utf-8\'))\nMANIFEST_SLICE = PAYLOAD[\'manifest_slice\']\nSHARD_INDEX = PAYLOAD[\'shard_index\']\nregister_sensitivity_dgps()\nregister_sensitivity_methods()\napply_method_registry({\'method_registry\': MANIFEST_SLICE[\'method_registry\']})\n\nOUTPUT = Path(\'shard_output\')\nOUTPUT.mkdir(exist_ok=True)\nPARQUET = OUTPUT / \'sensitivity_r50_results.parquet\'\nLOG = OUTPUT / \'execution_log.jsonl\'\nCACHE = Path(\'/tmp/wcf_sensitivity_r50_cache\') / f\'{SHARD_INDEX:02d}\'\nCACHE.mkdir(parents=True, exist_ok=True)\n\nrows = pq.read_table(PARQUET).to_pylist() if PARQUET.exists() else []\ngroups = {}\nfor row in rows:\n    groups.setdefault(row[\'cell_key\'], []).append(row)\nsuccessful = {key for key, values in groups.items()\n              if not any(row[\'status\'] == \'failed\' for row in values)}\nfailed = set(groups) - successful\nif failed:\n    rows = [row for row in rows if row[\'cell_key\'] not in failed]\nstarted = time.time()\nfor position, item in enumerate(MANIFEST_SLICE[\'cells\'], 1):\n    if item[\'cell_key\'] in successful:\n        continue\n    cell = Cell(**{key: value for key, value in item.items()\n                   if key not in (\'cell_key\', \'test_seed\')})\n    cell_rows = run_cell(cell, cache_directory=CACHE)\n    rows.extend(cell_rows)\n    temporary = PARQUET.with_suffix(\'.tmp\')\n    from wasserstein_causal_forests.g3.runner import write_rows\n    write_rows(rows, temporary)\n    os.replace(temporary, PARQUET)\n    with LOG.open(\'a\', encoding=\'utf-8\') as handle:\n        handle.write(json.dumps({\'cell_key\': cell.key,\n                                 \'status\': cell_rows[0][\'status\'],\n                                 \'n_rows\': len(cell_rows),\n                                 \'wall_seconds\': cell_rows[0][\'wall_seconds\'],\n                                 \'finished_at\': time.time()}) + \'\\n\')\n    print(f\'[{position}/{len(MANIFEST_SLICE["cells"])}] {cell.grid} {cell.dgp} \'\n          f\'K={cell.n_grid} M={cell.n_particles} n={cell.n_train} {cell.method} \'\n          f\'seed={cell.seed}: {cell_rows[0]["status"]}\', flush=True)\n    del cell_rows\n    gc.collect()\nprint(\'elapsed hours:\', round((time.time() - started) / 3600, 3))\n'
worker = Path(os.environ['WCF_SOURCE_ROOT']) / 'run_shard_worker.py'
worker.write_text(WORKER_SOURCE, encoding='utf-8')
environment = dict(os.environ)
environment['PYTHONUNBUFFERED'] = '1'
process = subprocess.Popen(
    [sys.executable, str(worker)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=environment,
)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    status = process.wait()
except KeyboardInterrupt:
    process.terminate()
    process.wait()
    raise
assert status == 0, f'run_shard_worker.py exited with status ' + str(status)
print('run_shard_worker.py finished cleanly')


In [ ]:
import os, subprocess, sys
from pathlib import Path
WORKER_SOURCE = '\\\nimport hashlib, json, platform, subprocess, sys, time\nfrom pathlib import Path\n\nimport pyarrow.parquet as pq\n\nPAYLOAD = json.loads(Path(\'wcf_sensitivity_r50_payload.json\').read_text(encoding=\'utf-8\'))\nMANIFEST_SLICE = PAYLOAD[\'manifest_slice\']\nSHARD_INDEX = PAYLOAD[\'shard_index\']\nSHARD_TOTAL = PAYLOAD[\'shard_total\']\nSOURCE_ARCHIVE_SHA256 = PAYLOAD[\'source_archive_sha256\']\n\nout = Path(\'shard_output\')\nparquet = out / \'sensitivity_r50_results.parquet\'\nframe = pq.read_table(parquet).to_pandas()\nexpected = {item[\'cell_key\'] for item in MANIFEST_SLICE[\'cells\']}\nobserved = set(frame.cell_key)\nassert observed == expected, (len(expected - observed), len(observed - expected))\nversions = {\'python\': sys.version, \'platform\': platform.platform()}\nfor name in (\'numpy\',\'scipy\',\'sklearn\',\'pandas\',\'pyarrow\'):\n    module = __import__(name)\n    versions[name] = module.__version__\nversions[\'R\'] = subprocess.run([\'Rscript\',\'-e\',\'cat(R.version.string)\'], capture_output=True, text=True, check=True).stdout\nversions[\'cran_drf\'] = subprocess.run([\'Rscript\',\'-e\',\'cat(as.character(packageVersion("drf")))\'], capture_output=True, text=True, check=True).stdout\nconfig = {\'shard_index\': SHARD_INDEX, \'shard_total\': SHARD_TOTAL,\n          \'manifest_contract_id\': MANIFEST_SLICE[\'manifest_contract_id\'],\n          \'manifest_checksum\': MANIFEST_SLICE[\'manifest_checksum\'],\n          \'estimator_source_hash\': MANIFEST_SLICE[\'estimator_source_hash\'],\n          \'evaluation_protocol_id\': MANIFEST_SLICE[\'evaluation_protocol_id\'],\n          \'source_archive_sha256\': SOURCE_ARCHIVE_SHA256, \'n_cells\': len(expected),\n          \'causal_drf_paper_repository\': PAYLOAD[\'causal_drf_paper_repository\'],\n          \'causal_drf_paper_commit\': PAYLOAD[\'causal_drf_paper_commit\'],\n          \'causal_drf_paper_files\': PAYLOAD[\'causal_drf_paper_files\'],\n          \'causal_clean_drf_commit\': PAYLOAD[\'causal_clean_drf_commit\'],\n          \'n_failed\': int(frame.loc[frame.status == \'failed\', \'cell_key\'].nunique()),\n          \'versions\': versions, \'completed_at\': time.time()}\n(out / \'manifest_slice.json\').write_text(json.dumps(MANIFEST_SLICE, indent=2), encoding=\'utf-8\')\n(out / \'completion.json\').write_text(json.dumps(config, indent=2), encoding=\'utf-8\')\ninventory = {path.name: hashlib.sha256(path.read_bytes()).hexdigest()\n             for path in out.iterdir() if path.is_file()}\n(out / \'sha256_inventory.json\').write_text(json.dumps(inventory, indent=2), encoding=\'utf-8\')\nprint(json.dumps(config, indent=2))\n'
worker = Path(os.environ['WCF_SOURCE_ROOT']) / 'finalize_shard_worker.py'
worker.write_text(WORKER_SOURCE, encoding='utf-8')
environment = dict(os.environ)
environment['PYTHONUNBUFFERED'] = '1'
process = subprocess.Popen(
    [sys.executable, str(worker)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=environment,
)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    status = process.wait()
except KeyboardInterrupt:
    process.terminate()
    process.wait()
    raise
assert status == 0, f'finalize_shard_worker.py exited with status ' + str(status)
print('finalize_shard_worker.py finished cleanly')


In [ ]:
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile
output_file = 'wcf_sensitivity_r50_shard_38_94c31a64332f.zip'
with ZipFile(output_file, 'w', ZIP_DEFLATED) as archive:
    for path in Path('shard_output').iterdir():
        if path.is_file():
            archive.write(path, path.name)
try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)
